In [1]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [2]:
df=pd.read_csv('loan_data.csv')
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [3]:
X=df.drop(columns='loan_status')
y=df.loan_status

In [4]:
xtrain,xtest,ytrain,ytest=train_test_split(X,y,train_size=0.8,random_state=42)

In [5]:
# num_cols=X.select_dtypes(include='number').columns
obj_cols=X.select_dtypes(include='object').columns

C:\Users\DELL\AppData\Local\Temp\ipykernel_14960\1865124758.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols=X.select_dtypes(include='object').columns


In [6]:
X[obj_cols].nunique()

person_gender                     2
person_education                  5
person_home_ownership             4
loan_intent                       6
previous_loan_defaults_on_file    2
dtype: int64

In [7]:
X['person_education'].unique()

<ArrowStringArray>
['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']
Length: 5, dtype: str

In [8]:
order=['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']

In [9]:
preprocessing=ColumnTransformer(
    transformers=[
        ('onehot_encoder',OneHotEncoder(handle_unknown='ignore'),obj_cols.drop('person_education')),
        ('ordinal_encoder',OrdinalEncoder(categories=[order],handle_unknown='use_encoded_value',unknown_value=-1),['person_education'])
    ],remainder='passthrough'
)



In [10]:
main_pipeline=Pipeline(
    steps=[
        ('preprocessing',preprocessing),
        ('model',LogisticRegression(random_state=42))
    ]
)

In [11]:
grid_search_cv = GridSearchCV(
    estimator=main_pipeline,
    param_grid={
        'model__C': [0.01, 0.1, 1, 10, 100],
        'model__penalty': ['l2'],
        'model__solver': ['lbfgs', 'liblinear'],
        'model__max_iter': [100, 200, 500]
    },
    cv=5,
    scoring='accuracy',
    verbose=10,
    n_jobs=-1
)

In [12]:
grid_search_cv.fit(xtrain,ytrain)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regre

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...], 'model__max_iter': [100, 200, ...], 'model__penalty': ['l2'], 'model__solver': ['lbfgs', 'liblinear']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fo

In [13]:
grid_search_cv.best_estimator_

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehot_encoder', ...), ('ordinal_encoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output o

In [14]:
grid_search_cv.best_params_

{'model__C': 0.01,
 'model__max_iter': 500,
 'model__penalty': 'l2',
 'model__solver': 'lbfgs'}

In [15]:
results=pd.DataFrame(grid_search_cv.cv_results_)
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__max_iter,param_model__penalty,param_model__solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,4.024523,0.320939,0.229115,0.108231,0.01,100,l2,lbfgs,"{'model__C': 0.01, 'model__max_iter': 100, 'mo...",0.857639,0.870972,0.826667,0.866250,0.863194,0.856944,0.015746,12
1,2.464551,1.007180,0.136544,0.078722,0.01,100,l2,liblinear,"{'model__C': 0.01, 'model__max_iter': 100, 'mo...",0.791944,0.795000,0.885972,0.836667,0.833333,0.828583,0.034210,19
2,4.583789,0.221965,0.138283,0.028520,0.01,200,l2,lbfgs,"{'model__C': 0.01, 'model__max_iter': 200, 'mo...",0.883611,0.890278,0.874167,0.880972,0.870278,0.879861,0.007044,8
3,2.014892,0.600153,0.128880,0.036681,0.01,200,l2,liblinear,"{'model__C': 0.01, 'model__max_iter': 200, 'mo...",0.791944,0.795000,0.885972,0.836667,0.833333,0.828583,0.034210,19
4,17.269884,1.369116,0.418462,0.122598,0.01,500,l2,lbfgs,"{'model__C': 0.01, 'model__max_iter': 500, 'mo...",0.886250,0.889028,0.884861,0.883472,0.885139,0.885750,0.001863,1
5,2.817802,1.144258,0.195732,0.073357,0.01,500,l2,liblinear,"{'model__C': 0.01, 'model__max_iter': 500, 'mo...",0.791944,0.795000,0.885972,0.836667,0.833333,0.828583,0.034210,19
6,5.694790,2.336324,0.340531,0.212408,0.10,100,l2,lbfgs,"{'model__C': 0.1, 'model__max_iter': 100, 'mod...",0.827500,0.862917,0.846389,0.861667,0.862500,0.852194,0.013816,13
7,3.711394,0.387322,0.225301,0.139194,0.10,100,l2,liblinear,"{'model__C': 0.1, 'model__max_iter': 100, 'mod...",0.791944,0.795000,0.871528,0.836389,0.833611,0.825694,0.029526,22
8,6.372355,0.646601,0.104179,0.028935,0.10,200,l2,lbfgs,"{'model__C': 0.1, 'model__max_iter': 200, 'mod...",0.866806,0.885556,0.881389,0.876944,0.883056,0.878750,0.006599,9
9,2.837124,0.880783,0.147074,0.065903,0.10,200,l2,liblinear,"{'model__C': 0.1, 'model__max_iter': 200, 'mod...",0.791944,0.795000,0.871528,0.836389,0.833611,0.825694,0.029526,22


In [16]:
results.sort_values(by='rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__max_iter,param_model__penalty,param_model__solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
4,17.269884,1.369116,0.418462,0.122598,0.01,500,l2,lbfgs,"{'model__C': 0.01, 'model__max_iter': 500, 'mo...",0.886250,0.889028,0.884861,0.883472,0.885139,0.885750,0.001863,1
10,11.131490,0.471768,0.117521,0.025589,0.10,500,l2,lbfgs,"{'model__C': 0.1, 'model__max_iter': 500, 'mod...",0.885556,0.885833,0.885556,0.880139,0.883194,0.884056,0.002179,2
16,13.702197,1.145438,0.296624,0.133306,1.00,500,l2,lbfgs,"{'model__C': 1, 'model__max_iter': 500, 'model...",0.885833,0.884583,0.884028,0.881111,0.882917,0.883694,0.001597,3
22,11.172259,0.263687,0.099640,0.026946,10.00,500,l2,lbfgs,"{'model__C': 10, 'model__max_iter': 500, 'mode...",0.885694,0.884583,0.883889,0.880278,0.883333,0.883556,0.001819,4
28,10.482849,0.573983,0.062537,0.014775,100.00,500,l2,lbfgs,"{'model__C': 100, 'model__max_iter': 500, 'mod...",0.884444,0.884583,0.883889,0.880139,0.883056,0.883222,0.001633,5
20,5.498605,0.135508,0.115841,0.028897,10.00,200,l2,lbfgs,"{'model__C': 10, 'model__max_iter': 200, 'mode...",0.885139,0.885417,0.880833,0.878056,0.883333,0.882556,0.002781,6
26,4.580719,0.622446,0.155195,0.032001,100.00,200,l2,lbfgs,"{'model__C': 100, 'model__max_iter': 200, 'mod...",0.885139,0.885417,0.880694,0.877222,0.883472,0.882389,0.003081,7
2,4.583789,0.221965,0.138283,0.028520,0.01,200,l2,lbfgs,"{'model__C': 0.01, 'model__max_iter': 200, 'mo...",0.883611,0.890278,0.874167,0.880972,0.870278,0.879861,0.007044,8
8,6.372355,0.646601,0.104179,0.028935,0.10,200,l2,lbfgs,"{'model__C': 0.1, 'model__max_iter': 200, 'mod...",0.866806,0.885556,0.881389,0.876944,0.883056,0.878750,0.006599,9
14,4.594002,0.123498,0.120840,0.032515,1.00,200,l2,lbfgs,"{'model__C': 1, 'model__max_iter': 200, 'model...",0.859028,0.885833,0.880139,0.877917,0.883472,0.877278,0.009521,10
